In [56]:
## Arxiv--research
## Toools creation
from langchain_community.tools import ArxivQueryRun,WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper,ArxivAPIWrapper


In [ ]:
api_wrapper_wiki=WikipediaAPIWrapper(top_k_results=1,doc_content_chars_max=250)
wiki=WikipediaQueryRun(api_wrapper=api_wrapper_wiki)
wiki.name

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [58]:
api_wrapper_arxiv=ArxivAPIWrapper(top_k_results=1,doc_content_chars_max=250)
arxiv=ArxivQueryRun(api_wrapper=api_wrapper_arxiv)
print(arxiv.name)

arxiv


In [59]:
tools=[wiki,arxiv]

In [60]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
import os
from dotenv import load_dotenv

In [61]:
google_api_key=os.getenv('GOOGLE_API_KEY')

In [62]:
loader=WebBaseLoader("https://docs.smith.langchain.com/")
embeddings=GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001",
# google_api_key=google_api_key
)
docs=loader.load()
document=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200).split_documents(docs)
vectordb=FAISS.from_documents(document,embeddings)
retriever=vectordb.as_retriever()
retriever

VectorStoreRetriever(tags=['FAISS', 'GoogleGenerativeAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001D83DCD7010>)

In [63]:
from langchain.tools.retriever import create_retriever_tool
retriever_tool=create_retriever_tool(retriever,"langsmith-search","Search any information about Langsmith ")

retriever_tool.name

'langsmith-search'

In [64]:
tools=[wiki,arxiv,retriever_tool]


In [65]:
tools

[WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from 'd:\\Langchain\\venv\\Lib\\site-packages\\wikipedia\\__init__.py'>, top_k_results=1, lang='en', load_all_available_meta=False, doc_content_chars_max=250)),
 ArxivQueryRun(api_wrapper=ArxivAPIWrapper(arxiv_search=<class 'arxiv.Search'>, arxiv_exceptions=(<class 'arxiv.ArxivError'>, <class 'arxiv.UnexpectedEmptyPageError'>, <class 'arxiv.HTTPError'>), top_k_results=1, ARXIV_MAX_QUERY_LENGTH=300, continue_on_failure=False, load_max_docs=100, load_all_available_meta=False, doc_content_chars_max=250)),
 Tool(name='langsmith-search', description='Search any information about Langsmith ', args_schema=<class 'langchain_core.tools.retriever.RetrieverInput'>, func=functools.partial(<function _get_relevant_documents at 0x000001D86EFCA700>, retriever=VectorStoreRetriever(tags=['FAISS', 'GoogleGenerativeAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001D83DCD7010>), docu

In [66]:
from langchain_groq import ChatGroq

groq_api_key=os.getenv('GROQ_API_KEY')
llm=ChatGroq(
        model='llama-3.3-70b-versatile',
        groq_api_key=groq_api_key
    )

In [67]:
from langchain import hub
prompt=hub.pull("hwchase17/openai-functions-agent")
prompt.messages

[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], template='You are a helpful assistant')),
 MessagesPlaceholder(variable_name='chat_history', optional=True),
 HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], template='{input}')),
 MessagesPlaceholder(variable_name='agent_scratchpad')]

In [68]:
## Agents
from langchain.agents import create_openai_tools_agent
agent=create_openai_tools_agent(llm,tools,prompt)
agent

RunnableAssign(mapper={
  agent_scratchpad: RunnableLambda(lambda x: format_to_openai_tool_messages(x['intermediate_steps']))
})
| ChatPromptTemplate(input_variables=['agent_scratchpad', 'input'], optional_variables=['chat_history'], input_types={'chat_history': typing.List[typing.Union[langchain_core.messages.ai.AIMessage, langchain_core.messages.human.HumanMessage, langchain_core.messages.chat.ChatMessage, langchain_core.messages.system.SystemMessage, langchain_core.messages.function.FunctionMessage, langchain_core.messages.tool.ToolMessage]], 'agent_scratchpad': typing.List[typing.Union[langchain_core.messages.ai.AIMessage, langchain_core.messages.human.HumanMessage, langchain_core.messages.chat.ChatMessage, langchain_core.messages.system.SystemMessage, langchain_core.messages.function.FunctionMessage, langchain_core.messages.tool.ToolMessage]]}, partial_variables={'chat_history': []}, metadata={'lc_hub_owner': 'hwchase17', 'lc_hub_repo': 'openai-functions-agent', 'lc_hub_commit_has

In [69]:
# Agent Excuter
from langchain.agents import AgentExecutor
agent_excutor=AgentExecutor(agent=agent,tools=tools,verbose=True)

In [70]:
agent_excutor.invoke({"input":"Tell me about Langsmith"})

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")



Invoking: `langsmith-search` with `{'query': 'Langsmith'}`


cause, and resolve them with LangSmith Engine.For terminology and core concepts, refer to Observability concepts. For trace pricing, retention, and limits, see Usage and billing.To set up a LangSmith instance, visit the Platform setup section to choose between cloud, hybrid, or self-hosted. All options include observability, evaluation, prompt engineering, and deployment.

LangSmith Observability - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentDocs by LangChain home pageMonitorSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangSmith ObservabilityOverviewEngineTraceDebugObserveLangSmith ObservabilityLangSmith Observability provides full visibility into your LLM application: from individual traces to production-wide performance metrics.LangSmith works with many frameworks and

{'input': 'Tell me about Langsmith',
 'output': 'LangSmith is a platform that provides observability and monitoring capabilities for large language models (LLMs). It offers features such as tracing, debugging, and performance metrics to help developers and users understand and improve the performance of their LLM applications. LangSmith works with various frameworks and providers, including OpenAI, Anthropic, and Vercel AI SDK, and provides integrations for popular tools like GitHub and Google. The platform also offers automated workflows, feedback collection, and issue detection and resolution capabilities. Users can sign up for a free account and create an API key to get started with LangSmith.'}

In [79]:
agent_excutor.invoke({"input":"What is Langchain"})


Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")



Invoking: `langsmith-search` with `{'query': 'Langchain'}`


LangSmith Observability - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentDocs by LangChain home pageMonitorSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangSmith ObservabilityOverviewEngineTraceDebugObserveLangSmith ObservabilityLangSmith Observability provides full visibility into your LLM application: from individual traces to production-wide performance metrics.LangSmith works with many frameworks and providers. Browse available integrations to connect your stack including OpenAI, Anthropic, CrewAI, Vercel AI SDK, Pydantic AI, and more.Get startedCreate an accountSign up at smith.langchain.com (no credit card required).
You can log in with Google, GitHub, or email.Create an API keyGo to your Settings page → API Keys → Create API Key.

cause, and resolve them with LangSm

{'input': 'What is Langchain',
 'output': 'Langchain is a platform that provides observability, evaluation, prompt engineering, and deployment for large language models (LLMs). It offers a range of tools and features to help developers monitor, debug, and improve the performance of their LLM applications. Langchain works with many frameworks and providers, including OpenAI, Anthropic, and CrewAI, and provides a free tier with no credit card required. It also offers a range of integrations, including a cloud, hybrid, or self-hosted platform, and provides features such as tracing, monitoring, and automation.'}

In [86]:
import arxiv

client = arxiv.Client()
search = arxiv.Search(id_list=["1706.03762"])

for paper in client.results(search):
    print(paper.title)
    break

Attention Is All You Need


In [88]:
import langchain
import langchain_community
import arxiv

print(langchain.__version__)
print(langchain_community.__version__)
print(arxiv.__version__)

0.2.17
0.2.19
4.0.0


In [84]:
agent_excutor.invoke({"input":"What's the paper 1706.03762 about?"})


Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")



Invoking: `arxiv` with `{'query': '1706.03762'}`




AttributeError: 'Search' object has no attribute 'results'

In [89]:
!pip uninstall arxiv -y
!pip install arxiv==2.1.3

Found existing installation: arxiv 4.0.0
Uninstalling arxiv-4.0.0:
  Successfully uninstalled arxiv-4.0.0
  Using cached feedparser-6.0.12-py3-none-any.whl.metadata (2.7 kB)
  Using cached sgmllib3k-1.0.0-py3-none-any.whl
Using cached feedparser-6.0.12-py3-none-any.whl (81 kB)

   ---------------------------------------- 0/4 [sgmllib3k]
  Attempting uninstall: requests
   ---------------------------------------- 0/4 [sgmllib3k]
    Found existing installation: requests 2.33.1
   ---------------------------------------- 0/4 [sgmllib3k]
    Uninstalling requests-2.33.1:
   ---------------------------------------- 0/4 [sgmllib3k]
      Successfully uninstalled requests-2.33.1
   ---------------------------------------- 0/4 [sgmllib3k]
   ---------- ----------------------------- 1/4 [requests]
   ---------- ----------------------------- 1/4 [requests]
   ---------- ----------------------------- 1/4 [requests]
   -------------------- ------------------- 2/4 [feedparser]
   -----------------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorboard 2.15.2 requires absl-py>=0.4, which is not installed.
tensorflow-intel 2.15.0 requires absl-py>=1.0.0, which is not installed.
tensorflow-intel 2.15.0 requires astunparse>=1.6.0, which is not installed.
tensorflow-intel 2.15.0 requires gast!=0.5.0,!=0.5.1,!=0.5.2,>=0.2.1, which is not installed.
tensorflow-intel 2.15.0 requires google-pasta>=0.1.1, which is not installed.
tensorflow-intel 2.15.0 requires h5py>=2.9.0, which is not installed.
tensorflow-intel 2.15.0 requires keras<2.16,>=2.15.0, which is not installed.
tensorflow-intel 2.15.0 requires libclang>=13.0.0, which is not installed.
tensorflow-intel 2.15.0 requires ml-dtypes~=0.2.0, which is not installed.
tensorflow-intel 2.15.0 requires opt-einsum>=2.3.2, which is not installed.
tensorflow-intel 2.15.0 requires termcolor>=1.1.0, which is not 

In [91]:
client = arxiv.Client()
search = arxiv.Search(id_list=["1706.03762"])

In [92]:
response = agent_excutor.invoke(
    {"input": "Tell me about LangSmith"}
)

print(response)

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")



Invoking: `langsmith-search` with `{'query': 'LangSmith information'}`


cause, and resolve them with LangSmith Engine.For terminology and core concepts, refer to Observability concepts. For trace pricing, retention, and limits, see Usage and billing.To set up a LangSmith instance, visit the Platform setup section to choose between cloud, hybrid, or self-hosted. All options include observability, evaluation, prompt engineering, and deployment.

LangSmith Observability - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentDocs by LangChain home pageMonitorSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangSmith ObservabilityOverviewEngineTraceDebugObserveLangSmith ObservabilityLangSmith Observability provides full visibility into your LLM application: from individual traces to production-wide performance metrics.LangSmith works with many fr

In [94]:
import arxiv

client = arxiv.Client()
search = arxiv.Search(id_list=["1706.03762"])

for paper in client.results(search):
    print(paper.title)

Attention Is All You Need


In [95]:
print(type(arxiv))

<class 'module'>


In [96]:
import arxiv

api_wrapper_arxiv = ArxivAPIWrapper(
    top_k_results=1,
    doc_content_chars_max=250
)

arxiv_tool = ArxivQueryRun(
    api_wrapper=api_wrapper_arxiv
)

In [97]:
print(type(arxiv_tool))

<class 'langchain_community.tools.arxiv.tool.ArxivQueryRun'>


In [98]:
print(arxiv_tool.run("1706.03762"))

AttributeError: 'Search' object has no attribute 'results'